# RAG 调试：Validation Error Matrix 查询纠正效果

**目标**：端到端验证新增的 few-shot RAG 条目能否被正确检索并纠正 LLM 生成代码中的三类错误：
1. scale 未从 inspect 结果中读取（硬编码 30）
2. `actual/predicted` 方向反转
3. `pred`/`ref` 字段混淆

**调试流程**：
```
环境初始化
 → OpenAI Embeddings 初始化
 → Chroma 集合构建（OpenAI-backed）
 → 导入 few-shot 条目
 → 相似度检索测试
 → KNOWLEDGE_PROMPT 全链路测试（含 LLM 调用）
 → CODE_GEN_PROMPT 对比测试（无 RAG vs 有 RAG）
```

In [ ]:

# ── 1. 环境初始化 ─────────────────────────────────────────────────────────────
import os
import sys
import uuid
from pathlib import Path

# 项目根路径
PROJECT_ROOT = Path("/Users/fred/Code/gee-agent")
sys.path.insert(0, str(PROJECT_ROOT))

# 加载 .env（dotenv 可选；也可直接 os.environ 赋值）
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=True)
    print("✓ .env 已加载")
except ImportError:
    # 如果没安装 python-dotenv，手动读取
    env_path = PROJECT_ROOT / ".env"
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())
    print("✓ .env 手动解析完成")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
print(f"OPENAI_API_KEY present: {bool(OPENAI_API_KEY)}")
print(f"Project root: {PROJECT_ROOT}")
# test

✓ .env 已加载
OPENAI_API_KEY present: True
Project root: /Users/fred/Code/gee-agent


In [4]:

# ── 2. OpenAI Embeddings 初始化 ──────────────────────────────────────────────
from openai import OpenAI

_oai = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-ada-002"

def get_openai_embeddings(texts: list[str]) -> list[list[float]]:
    """调用 OpenAI text-embedding-ada-002，返回嵌入向量列表。"""
    resp = _oai.embeddings.create(input=texts, model=EMBED_MODEL)
    return [item.embedding for item in resp.data]

# 烟雾测试
_test_vec = get_openai_embeddings(["validation error matrix kappa"])
print(f"✓ Embedding 维度: {len(_test_vec[0])}  (ada-002 应为 1536)")


✓ Embedding 维度: 1536  (ada-002 应为 1536)


In [5]:

# ── 3. Chroma 集合构建（专用于本调试 notebook，使用 OpenAI embeddings）─────────
import chromadb
from chromadb.config import Settings as ChromaSettings

CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
DEBUG_COLLECTION = "gee_kb_openai_debug"  # 独立集合，不影响生产 gee_kb

_chroma = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=ChromaSettings(anonymized_telemetry=False),
)

# 获取或创建调试集合（不指定 embedding_function，显式传入 embeddings）
_coll = _chroma.get_or_create_collection(
    DEBUG_COLLECTION,
    metadata={"description": "debug: OpenAI-backed RAG for validation task"},
)
print(f"✓ Chroma 集合 '{DEBUG_COLLECTION}'，当前文档数: {_coll.count()}")


✓ Chroma 集合 'gee_kb_openai_debug'，当前文档数: 0


In [6]:

# ── 4. 导入 RAG 条目 ──────────────────────────────────────────────────────────
# 将 gee_rag_data/ 下的所有 txt 文件按 "===" 分隔符切分后入库。
# 若文档已存在（去重检查），跳过重复导入。

RAG_DATA_DIR = PROJECT_ROOT / "gee_rag_data"
CHUNK_SEP = "==="
DEDUP_META_KEY = "source_file"

def _ingest_txt_file(path: Path, collection, dry_run: bool = False) -> int:
    """
    读取 txt 文件，按 `===` 分隔切块，过滤空块后批量写入 collection。
    返回实际写入数量。
    """
    raw = path.read_text(encoding="utf-8")
    chunks = [c.strip() for c in raw.split(CHUNK_SEP) if c.strip()]
    if not chunks:
        return 0

    # 去重：已有同 source_file 的条目则跳过
    existing = collection.get(where={DEDUP_META_KEY: path.name})
    if existing["ids"]:
        print(f"  ⤷ 跳过 {path.name}：已存在 {len(existing['ids'])} 条记录")
        return 0

    if dry_run:
        print(f"  [dry_run] {path.name}: {len(chunks)} chunks")
        return len(chunks)

    embeddings = get_openai_embeddings(chunks)
    ids = [f"{path.stem}_{uuid.uuid4().hex[:8]}" for _ in chunks]
    metadatas = [{DEDUP_META_KEY: path.name, "chunk_index": i} for i in range(len(chunks))]
    collection.add(documents=chunks, embeddings=embeddings, ids=ids, metadatas=metadatas)
    return len(chunks)


total_added = 0
for txt_file in sorted(RAG_DATA_DIR.glob("*.txt")):
    n = _ingest_txt_file(txt_file, _coll)
    print(f"  {txt_file.name}: +{n} chunks")
    total_added += n

print(f"\n✓ 本次新增 {total_added} 条，集合总量: {_coll.count()}")


  few_shot_validation_error_matrix.txt: +1 chunks
  gee_api_docs_for_rag.txt: +1902 chunks

✓ 本次新增 1903 条，集合总量: 1903


In [7]:

# ── 5. 相似度检索测试 ─────────────────────────────────────────────────────────
# 目标查询（来自用户的实际请求）
TARGET_QUERY = """\
我知道一个这样的数据：1. Samples for validation: projects/ee-hku-geog7310/assets/samples_LUMHK \
2. Land Utilization Image Source: projects/ee-hku-geog7310/assets/LUM_end2021_mode \
这是一个用来做validation的样本数据，帮我完成这个任务：Please use the land utilization map (2021) \
as the reference to validate the results (samples for validation). Try to construct the error matrix, \
and calculation the users' accuracy, producers' accuracy, overall accuracy as well as the kappa index.\
"""

K = 3  # 检索 top-K 条

query_vec = get_openai_embeddings([TARGET_QUERY])[0]
results = _coll.query(
    query_embeddings=[query_vec],
    n_results=K,
    include=["documents", "metadatas", "distances"],
)

print(f"=== 检索 Top-{K} 结果（距离越小越相关） ===\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    print(f"── Hit {i+1}  distance={dist:.4f}  source={meta.get('source_file','?')} chunk={meta.get('chunk_index','?')}")
    print(doc[:600])
    print()


=== 检索 Top-3 结果（距离越小越相关） ===

── Hit 1  distance=0.3063  source=few_shot_validation_error_matrix.txt chunk=0
## FEW_SHOT: Validation Error Matrix (Land Use 2021 as Reference)

Intent: execution
Task type: classification validation
Keywords: errorMatrix, kappa, overall accuracy, producers accuracy, users accuracy, sampleRegions, actual, predicted, scale

User task pattern:
"Use land utilization map (2021) as reference to validate sample results, then compute error matrix, OA, PA, UA, and kappa."

Assets in this case:
- Reference map image: projects/ee-hku-geog7310/assets/LUM_end2021_mode
- Validation samples: projects/ee-hku-geog7310/assets/samples_LUMHK

Critical rules (must follow):
1) Use inspect m

── Hit 2  distance=0.4335  source=gee_api_docs_for_rag.txt chunk=27
## ee.Algorithms.Landsat.TOA
Calibrates Landsat DN to TOA reflectance and brightness temperature for Landsat and similar data. For recently-acquired scenes calibration coefficients are extracted from the image metadata; f

In [8]:

# ── 6. 构造 kb_context 并注入 KNOWLEDGE_PROMPT ───────────────────────────────
# 完整复现 knowledge_base_lookup 的逻辑，但使用 OpenAI embeddings 的调试集合

def debug_kb_lookup(query: str, k: int = 3) -> str:
    """从调试 Chroma 集合检索，返回拼接文本（复现生产 knowledge_base_lookup 逻辑）。"""
    q_vec = get_openai_embeddings([query])[0]
    res = _coll.query(
        query_embeddings=[q_vec],
        n_results=k,
        include=["documents"],
    )
    docs = res["documents"][0] if res["documents"] else []
    if not docs:
        return "（未找到相关文档）"
    return "\n\n---\n\n".join(docs)


kb_context = debug_kb_lookup(TARGET_QUERY, k=3)

# 复现 KNOWLEDGE_PROMPT
KNOWLEDGE_PROMPT_TEMPLATE = """\
你是 Google Earth Engine (GEE) 助手。

请基于提供的参考知识回答用户问题：
- 如果参考内容可直接回答，优先引用参考内容中的关键信息。
- 如果参考不足，明确说明不确定点，并给出可执行的下一步建议。
- 回答保持简洁、准确，避免编造 API 或数据集细节。

参考知识：
{kb_context}

用户问题：
{query}
"""

knowledge_prompt = KNOWLEDGE_PROMPT_TEMPLATE.format(
    kb_context=kb_context,
    query=TARGET_QUERY,
)

print("=== KNOWLEDGE_PROMPT 内容（发送给 LLM 的完整 prompt）===\n")
print(knowledge_prompt)


=== KNOWLEDGE_PROMPT 内容（发送给 LLM 的完整 prompt）===

你是 Google Earth Engine (GEE) 助手。

请基于提供的参考知识回答用户问题：
- 如果参考内容可直接回答，优先引用参考内容中的关键信息。
- 如果参考不足，明确说明不确定点，并给出可执行的下一步建议。
- 回答保持简洁、准确，避免编造 API 或数据集细节。

参考知识：
## FEW_SHOT: Validation Error Matrix (Land Use 2021 as Reference)

Intent: execution
Task type: classification validation
Keywords: errorMatrix, kappa, overall accuracy, producers accuracy, users accuracy, sampleRegions, actual, predicted, scale

User task pattern:
"Use land utilization map (2021) as reference to validate sample results, then compute error matrix, OA, PA, UA, and kappa."

Assets in this case:
- Reference map image: projects/ee-hku-geog7310/assets/LUM_end2021_mode
- Validation samples: projects/ee-hku-geog7310/assets/samples_LUMHK

Critical rules (must follow):
1) Use inspect metadata to set sampling scale.
   - Preferred: use image nominal scale from inspect output.
   - Fallback: use projection nominal scale from the first band.
   - Do not hardcode 30 when source data is 1

In [10]:

# ── 7. LLM 调用：KNOWLEDGE_PROMPT 纠正效果 ──────────────────────────────────
# Jupyter 内核已有事件循环，直接用顶层 await，不要用 asyncio.run()
from backend.app.services.llm_client import chat_with_llm

print("=== LLM 回复（知识问答模式，RAG 增强后）===\n")
knowledge_reply = await chat_with_llm(knowledge_prompt)
print(knowledge_reply)


18:10:27 [DEBUG] gee_agent.llm: [LLM INPUT] model=gemini-3.1-flash-lite  prompt(5509 chars):
你是 Google Earth Engine (GEE) 助手。

请基于提供的参考知识回答用户问题：
- 如果参考内容可直接回答，优先引用参考内容中的关键信息。
- 如果参考不足，明确说明不确定点，并给出可执行的下一步建议。
- 回答保持简洁、准确，避免编造 API 或数据集细节。

参考知识：
## FEW_SHOT: Validation Error Matrix (Land Use 2021 as Reference)

Intent: execution
Task type: classification validation
Keywords: errorMatrix, kappa, overall accuracy, producers accuracy, users accuracy, sampleRegions, actual, predicted, scale

User task pattern:
"Use land utilization map (2021) as reference to validate sample results, then compute error matrix, OA, PA, UA, and kappa."

Assets in this case:
- Reference map image: projects/ee-hku-geog7310/assets/LUM_end2021_mode
- Validation samples: projects/ee-hku-geog7310/assets/samples_LUMHK

Critical rules (must follow):
1) Use inspect metadata to set sampling scale.
   - Preferred: use image nominal scale from inspect output.
   - Fallback: use projection nominal scale from the first band.


=== LLM 回复（知识问答模式，RAG 增强后）===



18:10:37 [INFO] gee_agent.llm: [LLM OUTPUT] model=gemini-3.1-flash-lite  elapsed=10.5s  2064 chars:
根据您提供的资产和任务需求，以下是使用 Google Earth Engine (GEE) Python API 进行验证的完整流程。该代码遵循了 GEE 验证任务的最佳实践，特别是关于**尺度（Scale）的获取**和**混淆矩阵字段映射**的规则。

### GEE 验证代码实现

```python
import ee

# 初始化 Earth Engine
ee.Initialize()

# 1. 加载资产
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")

# 2. 动态获取分辨率（关键规则：不要硬编码 scale）
# 获取影像第一个波段的投影信息，并提取标称分辨率
map_band = ee.String(lum_image.bandNames().get(0))
map_scale = lum_image.select([map_band]).projection().nominalScale()

# 3. 准备验证数据
# 将参考影像重命名为 'map_class' 以便在 errorMatrix 中明确区分
ref_map = lum_image.select([map_band]).rename("map_class")

# 将参考影像的值采样到样本点上
# 注意：确保 properties 包含样本中的预测字段（假设为 "pred"）
validated = ref_map.sampleRegions(
    collection=samples,
    properties=["pred"], 
    scale=map_scale,
    geometries=False,
)

# 4. 计算混淆矩阵及精度指标
# 关键规则：actual 是参考值 (map_class)，p

根据您提供的资产和任务需求，以下是使用 Google Earth Engine (GEE) Python API 进行验证的完整流程。该代码遵循了 GEE 验证任务的最佳实践，特别是关于**尺度（Scale）的获取**和**混淆矩阵字段映射**的规则。

### GEE 验证代码实现

```python
import ee

# 初始化 Earth Engine
ee.Initialize()

# 1. 加载资产
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")

# 2. 动态获取分辨率（关键规则：不要硬编码 scale）
# 获取影像第一个波段的投影信息，并提取标称分辨率
map_band = ee.String(lum_image.bandNames().get(0))
map_scale = lum_image.select([map_band]).projection().nominalScale()

# 3. 准备验证数据
# 将参考影像重命名为 'map_class' 以便在 errorMatrix 中明确区分
ref_map = lum_image.select([map_band]).rename("map_class")

# 将参考影像的值采样到样本点上
# 注意：确保 properties 包含样本中的预测字段（假设为 "pred"）
validated = ref_map.sampleRegions(
    collection=samples,
    properties=["pred"], 
    scale=map_scale,
    geometries=False,
)

# 4. 计算混淆矩阵及精度指标
# 关键规则：actual 是参考值 (map_class)，predicted 是样本中的预测值 (pred)
error_matrix = validated.errorMatrix(actual="map_class", predicted="pred")


In [11]:

# ── 8. CODE_GEN_PROMPT 对比测试（无 RAG vs 有 RAG）───────────────────────────
# 模拟 inspect 步骤已返回的典型元数据（与真实 asset 一致）
MOCK_INSPECT_IMAGE = {
    "status": "ok",
    "asset_id": "projects/ee-hku-geog7310/assets/LUM_end2021_mode",
    "bands": ["b1"],
    "scales": {"b1": 10.0},
}
MOCK_INSPECT_SAMPLES = {
    "status": "ok",
    "asset_id": "projects/ee-hku-geog7310/assets/samples_LUMHK",
    "property_names": ["pred", "ref", "system:index"],
    "feature_count": 312,
    "geometry_type": "Point",
}

from backend.app.sandbox.env_rules import SANDBOX_CONSTRAINTS_BLOCK

# ── 构造 context_section（复现 orchestrator._build_context_section）
def build_context_section(assets: dict) -> str:
    if not assets:
        return ""
    lines = ["已知数据上下文（由前序 inspect 步骤获得，必须使用这些实际字段名）："]
    for aid, meta in assets.items():
        lines.append(f"  Asset: {aid}")
        if meta.get("bands"):
            lines.append(f"    - 波段列表：{meta['bands']}")
        if meta.get("property_names"):
            lines.append(f"    - 属性字段（实际字段名）：{meta['property_names']}")
        if meta.get("feature_count") is not None:
            lines.append(f"    - 要素总数：{meta['feature_count']}")
        if meta.get("geometry_type"):
            lines.append(f"    - 几何类型：{meta['geometry_type']}")
        if meta.get("scales"):
            lines.append(f"    - 各波段分辨率（米）：{meta['scales']}")
    return "\n".join(lines)

context_section = build_context_section({
    MOCK_INSPECT_IMAGE["asset_id"]: {
        "bands": MOCK_INSPECT_IMAGE["bands"],
        "scales": MOCK_INSPECT_IMAGE["scales"],
    },
    MOCK_INSPECT_SAMPLES["asset_id"]: {
        "property_names": MOCK_INSPECT_SAMPLES["property_names"],
        "feature_count": MOCK_INSPECT_SAMPLES["feature_count"],
        "geometry_type": MOCK_INSPECT_SAMPLES["geometry_type"],
    },
})

STEP_DESCRIPTION = "构建误差矩阵，计算 OA、PA、UA、Kappa"

CODE_GEN_TEMPLATE = """\
你是一个 GEE 代码生成器，运行在 Python + earthengine-api 环境中。

用户总需求：
{query}

当前步骤任务：
{step_description}

{context_section}

{kb_section}

""" + SANDBOX_CONSTRAINTS_BLOCK + """
只输出可直接执行的 Python 代码块（用 ```python ... ``` 包裹），不要有额外解释。
"""

# 版本 A：无 RAG 注入
prompt_no_rag = CODE_GEN_TEMPLATE.format(
    query=TARGET_QUERY,
    step_description=STEP_DESCRIPTION,
    context_section=context_section,
    kb_section="",
)

# 版本 B：注入 RAG 检索结果
kb_block = (
    "【RAG 知识库参考 — 本任务相关最佳实践】\n"
    + debug_kb_lookup(TARGET_QUERY, k=2)
)
prompt_with_rag = CODE_GEN_TEMPLATE.format(
    query=TARGET_QUERY,
    step_description=STEP_DESCRIPTION,
    context_section=context_section,
    kb_section=kb_block,
)

print(f"Prompt A (无 RAG): {len(prompt_no_rag)} chars")
print(f"Prompt B (有 RAG): {len(prompt_with_rag)} chars")
print("\n── Prompt B 增量部分（RAG 注入块）──")
print(kb_block[:800])


Prompt A (无 RAG): 2585 chars
Prompt B (有 RAG): 6431 chars

── Prompt B 增量部分（RAG 注入块）──
【RAG 知识库参考 — 本任务相关最佳实践】
## FEW_SHOT: Validation Error Matrix (Land Use 2021 as Reference)

Intent: execution
Task type: classification validation
Keywords: errorMatrix, kappa, overall accuracy, producers accuracy, users accuracy, sampleRegions, actual, predicted, scale

User task pattern:
"Use land utilization map (2021) as reference to validate sample results, then compute error matrix, OA, PA, UA, and kappa."

Assets in this case:
- Reference map image: projects/ee-hku-geog7310/assets/LUM_end2021_mode
- Validation samples: projects/ee-hku-geog7310/assets/samples_LUMHK

Critical rules (must follow):
1) Use inspect metadata to set sampling scale.
   - Preferred: use image nominal scale from inspect output.
   - Fallback: use projection nominal scale from the first band.
   - Do not hardcod


In [12]:

# ── 9. LLM 对比：无 RAG 生成代码 ─────────────────────────────────────────────
print("=== 版本 A：无 RAG 注入的代码生成 ===\n")
reply_no_rag = await chat_with_llm(prompt_no_rag)
print(reply_no_rag)


18:13:28 [DEBUG] gee_agent.llm: [LLM INPUT] model=gemini-3.1-flash-lite  prompt(2585 chars):
你是一个 GEE 代码生成器，运行在 Python + earthengine-api 环境中。

用户总需求：
我知道一个这样的数据：1. Samples for validation: projects/ee-hku-geog7310/assets/samples_LUMHK 2. Land Utilization Image Source: projects/ee-hku-geog7310/assets/LUM_end2021_mode 这是一个用来做validation的样本数据，帮我完成这个任务：Please use the land utilization map (2021) as the reference to validate the results (samples for validation). Try to construct the error matrix, and calculation the users' accuracy, producers' accuracy, overall accuracy as well as the kappa index.

当前步骤任务：
构建误差矩阵，计算 OA、PA、UA、Kappa

已知数据上下文（由前序 inspect 步骤获得，必须使用这些实际字段名）：
  Asset: projects/ee-hku-geog7310/assets/LUM_end2021_mode
    - 波段列表：['b1']
    - 各波段分辨率（米）：{'b1': 10.0}
  Asset: projects/ee-hku-geog7310/assets/samples_LUMHK
    - 属性字段（实际字段名）：['pred', 'ref', 'system:index']
    - 要素总数：312
    - 几何类型：Point



【沙箱约束 — 严格遵守，违反将导致执行错误】
1. 禁止使用 geemap，禁止写 import geemap。
2. 可视化图层时使用预注入的 Map 对象：
  

=== 版本 A：无 RAG 注入的代码生成 ===



18:13:31 [INFO] gee_agent.llm: [LLM OUTPUT] model=gemini-3.1-flash-lite  elapsed=3.3s  964 chars:
```python
# 加载数据
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")

# 验证步骤：
# 根据已知上下文，samples 已经包含了 'pred' (预测值) 和 'ref' (参考值/真实值) 字段。
# 如果 samples 还没有从影像中采样，通常需要先进行采样。但根据题目描述，
# 样本数据似乎已经包含了 pred 和 ref 字段，因此可以直接使用 errorMatrix 方法。

# 1. 构建误差矩阵
# 使用 errorMatrix 方法，需要指定真实值字段名和预测值字段名
# 假设 'ref' 是真实值，'pred' 是预测值
error_matrix = samples.errorMatrix(actual='ref', predicted='pred')

# 2. 输出误差矩阵
print("Error Matrix:")
print(error_matrix.getInfo())

# 3. 计算 OA, PA, UA, Kappa
# Overall Accuracy (OA)
oa = error_matrix.accuracy()
print("Overall Accuracy:", oa.getInfo())

# Producers' Accuracy (PA)
pa = error_matrix.producersAccuracy()
print("Producers' Accuracy:", pa.getInfo())

# Users' Accuracy (UA)
ua = error_matrix.consumersAccuracy()
print("Users' Accuracy:", ua.getInfo())

# Kappa Index
kappa = e

```python
# 加载数据
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")

# 验证步骤：
# 根据已知上下文，samples 已经包含了 'pred' (预测值) 和 'ref' (参考值/真实值) 字段。
# 如果 samples 还没有从影像中采样，通常需要先进行采样。但根据题目描述，
# 样本数据似乎已经包含了 pred 和 ref 字段，因此可以直接使用 errorMatrix 方法。

# 1. 构建误差矩阵
# 使用 errorMatrix 方法，需要指定真实值字段名和预测值字段名
# 假设 'ref' 是真实值，'pred' 是预测值
error_matrix = samples.errorMatrix(actual='ref', predicted='pred')

# 2. 输出误差矩阵
print("Error Matrix:")
print(error_matrix.getInfo())

# 3. 计算 OA, PA, UA, Kappa
# Overall Accuracy (OA)
oa = error_matrix.accuracy()
print("Overall Accuracy:", oa.getInfo())

# Producers' Accuracy (PA)
pa = error_matrix.producersAccuracy()
print("Producers' Accuracy:", pa.getInfo())

# Users' Accuracy (UA)
ua = error_matrix.consumersAccuracy()
print("Users' Accuracy:", ua.getInfo())

# Kappa Index
kappa = error_matrix.kappa()
print("Kappa Index:", kappa.getInfo())
```


In [13]:

# ── 10. LLM 对比：有 RAG 注入的代码生成 ──────────────────────────────────────
print("=== 版本 B：RAG 增强后的代码生成 ===\n")
reply_with_rag = await chat_with_llm(prompt_with_rag)
print(reply_with_rag)


18:14:07 [DEBUG] gee_agent.llm: [LLM INPUT] model=gemini-3.1-flash-lite  prompt(6431 chars):
你是一个 GEE 代码生成器，运行在 Python + earthengine-api 环境中。

用户总需求：
我知道一个这样的数据：1. Samples for validation: projects/ee-hku-geog7310/assets/samples_LUMHK 2. Land Utilization Image Source: projects/ee-hku-geog7310/assets/LUM_end2021_mode 这是一个用来做validation的样本数据，帮我完成这个任务：Please use the land utilization map (2021) as the reference to validate the results (samples for validation). Try to construct the error matrix, and calculation the users' accuracy, producers' accuracy, overall accuracy as well as the kappa index.

当前步骤任务：
构建误差矩阵，计算 OA、PA、UA、Kappa

已知数据上下文（由前序 inspect 步骤获得，必须使用这些实际字段名）：
  Asset: projects/ee-hku-geog7310/assets/LUM_end2021_mode
    - 波段列表：['b1']
    - 各波段分辨率（米）：{'b1': 10.0}
  Asset: projects/ee-hku-geog7310/assets/samples_LUMHK
    - 属性字段（实际字段名）：['pred', 'ref', 'system:index']
    - 要素总数：312
    - 几何类型：Point

【RAG 知识库参考 — 本任务相关最佳实践】
## FEW_SHOT: Validation Error Matrix (Land Use 2021 as Referen

=== 版本 B：RAG 增强后的代码生成 ===



18:14:16 [INFO] gee_agent.llm: [LLM OUTPUT] model=gemini-3.1-flash-lite  elapsed=8.5s  1780 chars:
```python
import ee

# Initialize the Earth Engine API (assuming environment is already set up)
# ee.Initialize()

# Load assets
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")

# 1. Prepare Reference Data
# Get the nominal scale from the reference image metadata (10m)
map_band_name = ee.String(lum_image.bandNames().get(0))
map_scale = lum_image.select([map_band_name]).projection().nominalScale()

# Rename the reference band to 'actual' for clarity in the error matrix
ref_map = lum_image.select([map_band_name]).rename("actual")

# 2. Sample Reference Data at Sample Locations
# We use sampleRegions to extract the reference class ('actual') at the sample points.
# The sample collection already contains the predicted class in the 'pred' field.
validated = ref_map.sampleRegions(
    collec

```python
import ee

# Initialize the Earth Engine API (assuming environment is already set up)
# ee.Initialize()

# Load assets
lum_image = ee.Image("projects/ee-hku-geog7310/assets/LUM_end2021_mode")
samples = ee.FeatureCollection("projects/ee-hku-geog7310/assets/samples_LUMHK")

# 1. Prepare Reference Data
# Get the nominal scale from the reference image metadata (10m)
map_band_name = ee.String(lum_image.bandNames().get(0))
map_scale = lum_image.select([map_band_name]).projection().nominalScale()

# Rename the reference band to 'actual' for clarity in the error matrix
ref_map = lum_image.select([map_band_name]).rename("actual")

# 2. Sample Reference Data at Sample Locations
# We use sampleRegions to extract the reference class ('actual') at the sample points.
# The sample collection already contains the predicted class in the 'pred' field.
validated = ref_map.sampleRegions(
    collection=samples,
    properties=["pred"],  # We only need 'pred' from the original samples
    scale=m

In [14]:

# ── 11. 自动化检查：关键正确性指标 ───────────────────────────────────────────
# 对两份 LLM 输出做简单文本扫描，标记三类关键错误是否被修正

import re

def check_code_quality(code_text: str, label: str):
    print(f"\n{'='*50}")
    print(f"  质量检查: {label}")
    print(f"{'='*50}")

    issues = []
    passes = []

    # 1) actual/predicted 方向
    if re.search(r'actual\s*=\s*["\']map_class["\']', code_text) or \
       re.search(r'actual\s*=\s*ee\.String\(.*bandNames', code_text) or \
       re.search(r'actual\s*=\s*["\']b1["\']', code_text):
        passes.append("✓ actual 字段指向参考图层（正确）")
    else:
        issues.append("✗ actual 字段未明确指向参考图层（可能反转）")

    if re.search(r'predicted\s*=\s*["\']pred["\']', code_text):
        passes.append("✓ predicted 使用样本 pred 字段（正确）")
    else:
        issues.append("✗ predicted 未使用 pred 字段")

    # 2) scale 使用
    if re.search(r'scale\s*=\s*10', code_text) or \
       re.search(r'nominalScale\(\)', code_text) or \
       re.search(r'map_scale', code_text):
        passes.append("✓ scale 基于影像分辨率（10m 或动态获取）")
    elif re.search(r'scale\s*=\s*30', code_text):
        issues.append("✗ scale 硬编码为 30（影像实际为 10m）")
    else:
        issues.append("⚠ 未找到明确 scale 参数")

    # 3) 避免 ref 被误当 predicted
    if re.search(r'predicted\s*=\s*["\']ref["\']', code_text):
        issues.append("✗ predicted 使用了 ref 字段（语义错误：ref 是参考标签）")

    for p in passes:
        print(f"  {p}")
    for i in issues:
        print(f"  {i}")
    if not issues:
        print("  → 全部检查通过 🎉")
    else:
        print(f"  → 发现 {len(issues)} 个问题")

check_code_quality(reply_no_rag, "版本 A: 无 RAG")
check_code_quality(reply_with_rag, "版本 B: RAG 增强")



  质量检查: 版本 A: 无 RAG
  ✓ predicted 使用样本 pred 字段（正确）
  ✗ actual 字段未明确指向参考图层（可能反转）
  ⚠ 未找到明确 scale 参数
  → 发现 2 个问题

  质量检查: 版本 B: RAG 增强
  ✓ predicted 使用样本 pred 字段（正确）
  ✓ scale 基于影像分辨率（10m 或动态获取）
  ✗ actual 字段未明确指向参考图层（可能反转）
  → 发现 1 个问题
